# Notebook 01 — Data Cleaning
**Proyek**: Analisis Demografi Provinsi Indonesia  
**Sumber**: BPS Indonesia (bps.go.id)  
**Deskripsi**: Membersihkan dan menstandardisasi dataset demografi gabungan provinsi Indonesia sebelum analisis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data

In [ ]:
RAW_PATH = '../data/raw/demografi_provinsi_indonesia_merged.csv'
META_PATH = '../data/raw/metadata_demografi_provinsi_indonesia.csv'

df_raw = pd.read_csv(RAW_PATH)
df_meta = pd.read_csv(META_PATH)

print(f'Shape raw: {df_raw.shape}')
print(f'\nKolom: {df_raw.columns.tolist()}')
df_raw.head(5)

## 2. Inspeksi Awal

In [ ]:
print('=== INFO TIPE DATA ===')
print(df_raw.dtypes)
print()
print('=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])
print()
print(f'=== DUPLIKAT ===')
dupes = df_raw[df_raw['provinsi'].duplicated(keep=False)]
print(f'Baris duplikat: {len(dupes)}')
print(dupes[['id','provinsi','catatan']].sort_values('provinsi').to_string())

**Temuan**:
- Terdapat 8 baris duplikat untuk Gorontalo (kemungkinan dari penggabungan file yang berbeda)
- Beberapa kolom laju historis bertipe `object` karena ada nilai non-numerik (`'-'`, `'2,36 1'`)
- Missing values terbanyak pada indikator SP2020 untuk provinsi pemekar baru

## 3. Deduplikasi

In [ ]:
# Keputusan: ambil baris pertama untuk setiap provinsi
# Alasan: baris pertama memiliki data paling lengkap dan catatan NaN (= data bersih)
df = df_raw.drop_duplicates(subset='provinsi', keep='first').copy()
df = df.reset_index(drop=True)

print(f'Sebelum deduplikasi : {len(df_raw)} baris')
print(f'Setelah deduplikasi  : {len(df)} baris (38 provinsi)')

## 4. Cleaning Kolom Laju Historis (string → float)

In [ ]:
laju_str_cols = [
    'laju_1971_1980', 'laju_1980_1990', 'laju_1990_2000', 'laju_2000_2010',
    'laju_2010_2016', 'laju_2010_2017', 'laju_2010_2018',
    'laju_2010_2019', 'laju_2010_2020', 'laju_2020_2021', 'laju_2020_2022'
]

def clean_laju(val):
    """Membersihkan nilai laju: hapus suffix teks, ganti koma desimal, set '-' jadi NaN."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ['-', '', 'NA', 'N/A']:
        return np.nan
    # Hapus karakter non-numerik di akhir (misal: '2,36 1' → '2.36')
    s = s.replace(',', '.').split()[0]
    try:
        return float(s)
    except ValueError:
        return np.nan

for col in laju_str_cols:
    df[col] = df[col].apply(clean_laju)

# kepadatan_per_km2_2021 juga string
df['kepadatan_per_km2_2021'] = df['kepadatan_per_km2_2021'].apply(clean_laju)

print('Cek tipe data setelah cleaning:')
print(df[laju_str_cols].dtypes)

## 5. Standardisasi Nama Provinsi

In [ ]:
df['provinsi'] = df['provinsi'].str.strip().str.upper()

# Tambah label pulau untuk segmentasi analisis
JAWA = {'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'DI YOGYAKARTA', 'BANTEN'}
SUMATERA = {'ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'KEPULAUAN RIAU',
            'JAMBI', 'BENGKULU', 'SUMATERA SELATAN', 'KEPULAUAN BANGKA BELITUNG', 'LAMPUNG'}
KALIMANTAN = {'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN',
              'KALIMANTAN TIMUR', 'KALIMANTAN UTARA'}
SULAWESI = {'SULAWESI UTARA', 'GORONTALO', 'SULAWESI TENGAH', 'SULAWESI BARAT',
            'SULAWESI SELATAN', 'SULAWESI TENGGARA'}
PAPUA_NTB = {'PAPUA', 'PAPUA BARAT', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN',
             'PAPUA BARAT DAYA', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR'}

def assign_pulau(prov):
    if prov in JAWA: return 'Jawa'
    if prov in SUMATERA: return 'Sumatera'
    if prov in KALIMANTAN: return 'Kalimantan'
    if prov in SULAWESI: return 'Sulawesi'
    if prov in PAPUA_NTB: return 'Papua/NTT/NTB'
    return 'Lainnya'

df['pulau'] = df['provinsi'].apply(assign_pulau)
print(df.groupby('pulau').size().sort_values(ascending=False))

## 6. Validasi Final

In [ ]:
print(f'Jumlah provinsi unik : {df["provinsi"].nunique()}')
print(f'Total penduduk 2026  : {df["jumlah_penduduk_ribu_2026"].sum()/1000:.1f} juta jiwa')
print(f'Distribusi total     : {df["distribusi_pct_2026"].sum():.1f}% (harus ~100)')
print()
print('Missing values per kolom utama:')
key_cols = ['jumlah_penduduk_ribu_2026','laju_pertumbuhan_pct_2026','kepadatan_per_km2_2026',
            'laju_2020_2024','imr_per1000_2020','cbr_per1000_2020','tfr_2020']
print(df[key_cols].isnull().sum())

## 7. Simpan Data Processed

In [ ]:
OUT_PATH = '../data/processed/demografi_clean.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df.shape}')
df.head(3)